# 03c -- Student Model Evaluation (FIXED V2)

Evaluates the English-to-Swahili student Transformer produced by `03b_train_FIXED_V2.ipynb`.

**Expected checkpoint run name:** `student_beam_M1_optA_fixed_v2`

Checkpoint search priority:
1. `student_beam_M1_optA_fixed_v2_best_chrf.pt`
2. `student_beam_M1_optA_fixed_v2_best_val.pt`
3. `student_beam_M1_optA_fixed_v2_latest.pt`
4. Any `*fixed_v2*best*.pt` match

**Fixes applied vs original 03c_evaluate:**
- Separate greedy decoder (debugging baseline)
- Beam search: never early-stops while unfinished beams remain
- Pre-evaluation health gate (empty%, avg token len, unique ratio)
- SHA-256 tokenizer consistency check against checkpoint hash
- Resumable JSONL predictions with atomic final write
- SacreBLEU `tokenize='flores200'`, chrF++ `word_order=2`
- Dynamic path discovery; no hard-coded dataset slugs

**Platform:** Kaggle T4 GPU / PyTorch 2.x / Python 3.12

In [1]:
# Install only missing packages.
# Kaggle normally already includes torch, pandas, and tqdm.
import importlib.util
import subprocess
import sys
import pickle

_required = {
    "sentencepiece": "sentencepiece",
    "sacrebleu": "sacrebleu",
    "pandas": "pandas",
    "tqdm": "tqdm",
}

_missing = [pkg for pkg, mod in _required.items()
            if importlib.util.find_spec(mod) is None]

if _missing:
    print("Installing missing packages:", _missing)
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", *_missing]
        )
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Package installation failed. On Kaggle, enable Internet temporarily "
            "or add the missing wheel/package as a dataset."
        ) from exc
else:
    print("Required packages are already installed.")

Installing missing packages: ['sacrebleu']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 2.7 MB/s eta 0:00:00


In [2]:
import gc, hashlib, json, math, os, platform, re, sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import sacrebleu
import sentencepiece as spm
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

# ── Evaluation configuration ────────────────────────────────────────────
DATASET            = "beam_M10"    # beam_M1 | beam_M10 | top_p_M10 | top_k_M10 | dbs_M10 | mbr_M10
MODEL_SIZE         = "A"          # A or B
BEAM_SIZE          = 5
LENGTH_PENALTY     = 0.6
MIN_DECODE_TOKENS  = 1
MAX_DECODE_TOKENS  = None         # None = use checkpoint MAX_LENGTH
BLEU_TOKENIZER     = "flores200"
EVALUATE_DEV       = True
EVALUATE_DEVTEST   = True
EVAL_LIMIT         = None         # set to small int for quick test
RESUME_PARTIAL     = True
ALLOW_FAILED_HEALTH_CHECK = False  # only override for debugging
CHECKPOINT_OVERRIDE = None         # set to explicit path to bypass discovery

# ── Device ──────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python  :", sys.version)
print("PyTorch :", torch.__version__)
print("Platform:", platform.platform())
print("Device  :", device)
if device.type == "cuda":
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    : {:.1f} GB".format(
        torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("WARNING: Full beam-search evaluation on CPU will be very slow.")

Python  : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch : 2.10.0+cu128
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
Device  : cuda
GPU     : Tesla T4
VRAM    : 15.6 GB


In [3]:
# ── Path setup ──────────────────────────────────────────────────────────
IS_KAGGLE  = Path("/kaggle/working").exists()
WORK_ROOT  = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
RESULTS_DIR = WORK_ROOT / "results"
PREDS_DIR   = RESULTS_DIR / "predictions"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PREDS_DIR.mkdir(parents=True, exist_ok=True)

# Build ordered search roots: cwd first, then /kaggle/input recursively
_search_roots: List[Path] = []
for _r in [WORK_ROOT, Path.cwd()]:
    if _r.exists():
        _search_roots.append(_r.resolve())
_kaggle_input = Path("/kaggle/input")
if _kaggle_input.exists():
    _search_roots.append(_kaggle_input.resolve())
SEARCH_ROOTS: List[Path] = list(dict.fromkeys(_search_roots))

BASE_RUN_NAME = "student_{}_opt{}_fixed_v2".format(DATASET, MODEL_SIZE)

def _all_matches(pattern: str) -> List[Path]:
    found: List[Path] = []
    for root in SEARCH_ROOTS:
        try:
            found.extend(p.resolve() for p in root.rglob(pattern) if p.is_file())
        except (OSError, PermissionError):
            continue
    return list(dict.fromkeys(found))

def _checkpoint_rank(path: Path) -> Tuple[int, int, int, int, str]:
    """Lower tuple = higher priority."""
    name = path.name
    # Prefer a folder containing the dataset name
    dataset_folder_rank = 0 if DATASET in path.parts else 1
    # Prefer best_chrf > best_val > latest > other
    if "_best_chrf" in name:
        name_rank = 0
    elif "_best_val" in name:
        name_rank = 1
    elif "_latest" in name:
        name_rank = 3
    elif "best" in name.lower():
        name_rank = 2
    else:
        name_rank = 4
    # latest should NEVER beat best
    latest_rank = 1 if "latest" in name.lower() else 0
    # Prefer /kaggle/working over /kaggle/input
    working_rank = 0 if str(path).startswith(str(WORK_ROOT.resolve())) else 1
    return (dataset_folder_rank, latest_rank, name_rank, working_rank, str(path))

def find_checkpoint() -> Path:
    if CHECKPOINT_OVERRIDE is not None:
        _p = Path(CHECKPOINT_OVERRIDE).expanduser().resolve()
        if not _p.is_file():
            raise FileNotFoundError("CHECKPOINT_OVERRIDE does not exist: {}".format(_p))
        print("Selected checkpoint (override):", _p)
        return _p

    _priority_patterns = [
        "{}_best_chrf.pt".format(BASE_RUN_NAME),
        "{}_best_val.pt".format(BASE_RUN_NAME),
        "{}_latest.pt".format(BASE_RUN_NAME),
        "*fixed_v2*best*.pt",
    ]

    _candidates: List[Path] = []
    for pat in _priority_patterns:
        _candidates.extend(_all_matches(pat))
    _candidates = list(dict.fromkeys(_candidates))

    if not _candidates:
        _all_pt = _all_matches("*.pt")
        _preview = "\n".join("  - {}".format(p) for p in _all_pt[:30])
        raise FileNotFoundError(
            "No checkpoint found for {}.\n"
            "Expected names like:\n"
            "  {}_best_chrf.pt\n"
            "  {}_best_val.pt\n\n"
            "All .pt files found:\n{}".format(
                BASE_RUN_NAME, BASE_RUN_NAME, BASE_RUN_NAME,
                _preview or "  (none)"
            )
        )

    # Rank and sort
    _ranked = sorted(_candidates, key=_checkpoint_rank)
    _best = _ranked[0]

    print("Checkpoint candidates found ({}):".format(len(_ranked)))
    for _p in _ranked:
        _score = _checkpoint_rank(_p)
        _marker = "  [SELECTED]" if _p == _best else "          "
        _reason = "best_chrf" if "_best_chrf" in _p.name else (
                  "best_val"  if "_best_val"  in _p.name else (
                  "latest"    if "_latest"     in _p.name else "other_best"))
        print("{} {} (rank={}, reason={})".format(_marker, _p, _score[:4], _reason))

    _reason_best = ("best_chrf" if "_best_chrf" in _best.name else
                    "best_val"  if "_best_val"  in _best.name else
                    "latest"    if "_latest"     in _best.name else "other_best")
    print("\nSelected checkpoint: {} (reason: {})".format(_best, _reason_best))
    return _best

CKPT_BEST = find_checkpoint()
RUN_DIR   = CKPT_BEST.parent
RUN_ID    = CKPT_BEST.stem
print("Run directory :", RUN_DIR)
print("Results dir   :", RESULTS_DIR)

Checkpoint candidates found (5):
  [SELECTED] /kaggle/input/datasets/nirmitmistry/beamm10/beam_M10/student_beam_M10_optA_fixed_v2_best_chrf.pt (rank=(0, 0, 0, 1), reason=best_chrf)
           /kaggle/input/datasets/nirmitmistry/beamm10/beam_M10/student_beam_M10_optA_fixed_v2_best_val.pt (rank=(0, 0, 1, 1), reason=best_val)
           /kaggle/input/datasets/nirmitmistry/beamm10/beam_M10/student_beam_M10_optA_fixed_v2_latest.pt (rank=(0, 1, 3, 1), reason=latest)
           /kaggle/input/datasets/nirmitmistry/evalaution/beam-m1-eval/beam_M1/student_beam_M1_optA_fixed_v2_best_chrf.pt (rank=(1, 0, 0, 1), reason=best_chrf)
           /kaggle/input/datasets/nirmitmistry/evalaution/beam-m1-eval/beam_M1/student_beam_M1_optA_fixed_v2_best_val.pt (rank=(1, 0, 1, 1), reason=best_val)

Selected checkpoint: /kaggle/input/datasets/nirmitmistry/beamm10/beam_M10/student_beam_M10_optA_fixed_v2_best_chrf.pt (reason: best_chrf)
Run directory : /kaggle/input/datasets/nirmitmistry/beamm10/beam_M10
Results d

In [4]:
# ── Helpers ─────────────────────────────────────────────────────────────
def _common_prefix_depth(a: Path, b: Path) -> int:
    depth = 0
    for left, right in zip(a.parts, b.parts):
        if left != right:
            break
        depth += 1
    return depth

def _choose_file(
    exact_names: Sequence[str],
    preferred_dirs: Sequence[Path],
    description: str,
) -> Path:
    """Find a file, preferring locations near RUN_DIR."""
    for directory in preferred_dirs:
        for name in exact_names:
            _cand = directory / name
            if _cand.is_file():
                return _cand.resolve()
    _candidates: List[Path] = []
    for name in exact_names:
        _candidates.extend(_all_matches(name))
    _candidates = list(dict.fromkeys(_candidates))
    if not _candidates:
        raise FileNotFoundError(
            "Could not find {}. Looked for: {}".format(description, ", ".join(exact_names))
        )
    _candidates.sort(key=lambda p: (
        -_common_prefix_depth(p.parent, RUN_DIR),
        0 if DATASET in p.parts else 1,
        len(p.parts),
        str(p),
    ))
    _chosen = _candidates[0]
    print("{} candidates:".format(description))
    for _p in _candidates:
        print("  {} {}".format("[SELECTED]" if _p == _chosen else "         ", _p))
    return _chosen

# ── Locate vocab_info.json ───────────────────────────────────────────────
VOCAB_INFO_PATH = _choose_file(
    ["vocab_info.json"],
    [RUN_DIR, RUN_DIR.parent, WORK_ROOT / "notebooks" / "models"],
    "vocab_info.json",
)

# ── Locate SentencePiece model ───────────────────────────────────────────
SPM_MODEL_PATH = _choose_file(
    ["shared_spm.model", "spm_joint.model", "spm_model.model"],
    [RUN_DIR, VOCAB_INFO_PATH.parent, RUN_DIR.parent],
    "SentencePiece model",
)

# ── Load vocab_info ──────────────────────────────────────────────────────
with open(VOCAB_INFO_PATH, "r", encoding="utf-8") as _fv:
    _vi = json.load(_fv)

_required_keys = {"vocab_size", "pad_id", "bos_id", "eos_id", "max_length"}
_missing_keys  = sorted(_required_keys - set(_vi))
if _missing_keys:
    raise KeyError("{} is missing required keys: {}".format(VOCAB_INFO_PATH, _missing_keys))

VOCAB_SIZE  = int(_vi["vocab_size"])
PAD_ID      = int(_vi["pad_id"])
BOS_ID      = int(_vi["bos_id"])
EOS_ID      = int(_vi["eos_id"])
MAX_LENGTH  = int(_vi["max_length"])

# ── Load SentencePiece ───────────────────────────────────────────────────
sp = spm.SentencePieceProcessor(model_file=str(SPM_MODEL_PATH))

# ── Validate ─────────────────────────────────────────────────────────────
assert sp.get_piece_size() == VOCAB_SIZE, \
    "Tokenizer vocab {} != expected {}".format(sp.get_piece_size(), VOCAB_SIZE)
assert sp.bos_id() == BOS_ID, \
    "BOS mismatch: tokenizer={} vocab_info={}".format(sp.bos_id(), BOS_ID)
assert sp.eos_id() == EOS_ID, \
    "EOS mismatch: tokenizer={} vocab_info={}".format(sp.eos_id(), EOS_ID)

print("\nTokenizer validated OK")
print("  vocab_info :", VOCAB_INFO_PATH)
print("  spm model  :", SPM_MODEL_PATH)
print("  vocab={} PAD={} BOS={} EOS={} MAX_LENGTH={}".format(
    VOCAB_SIZE, PAD_ID, BOS_ID, EOS_ID, MAX_LENGTH))


Tokenizer validated OK
  vocab_info : /kaggle/input/datasets/nirmitmistry/beamm10/beam_M10/vocab_info.json
  spm model  : /kaggle/input/datasets/nirmitmistry/beamm10/beam_M10/shared_spm.model
  vocab=32000 PAD=0 BOS=2 EOS=3 MAX_LENGTH=128


In [5]:
# ── Safe loader ──────────────────────────────────────────────────────────
# ── Safe loader ──────────────────────────────────────────────────────────
def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    # Catch the UnpicklingError PyTorch 2.6 throws when blocking Numpy
    except (TypeError, RuntimeError, pickle.UnpicklingError): 
        print("Falling back to weights_only=False...")
        return torch.load(path, map_location=map_location, weights_only=False)

checkpoint = safe_torch_load(CKPT_BEST, map_location="cpu")
if not isinstance(checkpoint, dict):
    raise TypeError("Checkpoint must be a dict, got {}".format(type(checkpoint)))

# ── Extract state_dict ───────────────────────────────────────────────────
if "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
elif checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):
    state_dict = checkpoint
else:
    raise KeyError("Checkpoint contains neither 'model_state_dict' nor 'state_dict'")

# Strip DataParallel 'module.' prefix if present
if state_dict and all(k.startswith("module.") for k in state_dict):
    state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

# ── Extract model_cfg ────────────────────────────────────────────────────
_DEFAULT_CFGS = {
    "A": dict(d_model=512, nhead=8, num_encoder_layers=6,
               num_decoder_layers=6, d_ff=2048, dropout=0.3),
    "B": dict(d_model=128, nhead=4, num_encoder_layers=2,
               num_decoder_layers=2, d_ff=512, dropout=0.1),
}
saved_cfg = checkpoint.get("model_cfg", checkpoint.get("model_config"))
if saved_cfg is None:
    print("WARNING: Checkpoint has no model_cfg; using known Option {} defaults.".format(MODEL_SIZE))
    saved_cfg = _DEFAULT_CFGS[MODEL_SIZE]
if not isinstance(saved_cfg, dict):
    raise TypeError("model_cfg must be a dict")
_allowed = {"d_model","nhead","num_encoder_layers","num_decoder_layers","d_ff","dropout"}
_extra = sorted(set(saved_cfg) - _allowed)
if _extra:
    print("Ignoring extra model_cfg keys:", _extra)
saved_cfg = {k: saved_cfg[k] for k in _allowed if k in saved_cfg}
_missing_cfg = sorted(_allowed - set(saved_cfg))
if _missing_cfg:
    raise KeyError("model_cfg is missing: {}".format(_missing_cfg))

# ── Validate shapes ──────────────────────────────────────────────────────
if "embedding.weight" not in state_dict:
    raise KeyError("Checkpoint missing 'embedding.weight'")
if "pos_enc.pe" not in state_dict:
    raise KeyError("Checkpoint missing 'pos_enc.pe'")

_ckpt_vocab, _ckpt_d = state_dict["embedding.weight"].shape
checkpoint_positional_len = int(state_dict["pos_enc.pe"].shape[1])

if _ckpt_vocab != VOCAB_SIZE:
    raise ValueError(
        "Checkpoint vocab {} != tokenizer vocab {}".format(_ckpt_vocab, VOCAB_SIZE))
if _ckpt_d != int(saved_cfg["d_model"]):
    raise ValueError(
        "Checkpoint d_model {} != model_cfg d_model {}".format(_ckpt_d, saved_cfg["d_model"]))

# ── SHA-256 tokenizer check ───────────────────────────────────────────────
if "tokenizer_sha256" in checkpoint:
    _spm_bytes = SPM_MODEL_PATH.read_bytes()
    _spm_hash  = hashlib.sha256(_spm_bytes).hexdigest()
    if _spm_hash != checkpoint["tokenizer_sha256"]:
        print("WARNING: Tokenizer SHA-256 mismatch."
              " checkpoint={} file={}".format(
                  checkpoint["tokenizer_sha256"][:16], _spm_hash[:16]))
        print("  The model may have been trained with a different SPM binary."
              " Evaluation can proceed but scores may be slightly off.")
    else:
        print("Tokenizer SHA-256 OK.")

# ── Model definition (identical to 03b_train_FIXED_V2) ───────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.max_len = max_len
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        assert x.size(1) <= self.max_len, \
            "seq len {} > max_len {}".format(x.size(1), self.max_len)
        return self.dropout(x + self.pe[:, :x.size(1)])

class StudentTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_encoder_layers,
                 num_decoder_layers, d_ff, dropout, max_len=512, pad_id=0):
        super().__init__()
        self.d_model = d_model
        self.pad_id  = pad_id
        self.embedding   = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc     = PositionalEncoding(d_model, dropout, max_len)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=d_ff, dropout=dropout, batch_first=True,
        )
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.output_proj.weight = self.embedding.weight

    def pad_mask(self, ids):
        return ids == self.pad_id

    def causal_mask(self, length, device_):
        return torch.triu(
            torch.ones(length, length, dtype=torch.bool, device=device_), diagonal=1)

    def forward(self, src, tgt):
        scale   = self.d_model ** 0.5
        src_emb = self.pos_enc(self.embedding(src) * scale)
        tgt_emb = self.pos_enc(self.embedding(tgt) * scale)
        out = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=self.causal_mask(tgt.size(1), src.device),
            src_key_padding_mask=self.pad_mask(src),
            tgt_key_padding_mask=self.pad_mask(tgt),
            memory_key_padding_mask=self.pad_mask(src),
        )
        return self.output_proj(out)

# ── Build and load ────────────────────────────────────────────────────────
model = StudentTransformer(
    vocab_size=VOCAB_SIZE,
    max_len=checkpoint_positional_len,
    pad_id=PAD_ID,
    **saved_cfg,
)
try:
    model.load_state_dict(state_dict, strict=True)
except RuntimeError as exc:
    raise RuntimeError(
        "Model definition does not exactly match checkpoint. "
        "Do not evaluate with a non-strict load."
    ) from exc

model = model.to(device)
model.eval()

# Report
_ckpt_epoch  = checkpoint.get("epoch")
_metric_name = None
_metric_val  = None
for _k in ("best_chrf", "best_dev_chrf", "best_val_loss", "best_val_chrf", "best_metric"):
    if _k in checkpoint:
        _metric_name = _k
        _metric_val  = checkpoint[_k]
        break

_n_params = sum(p.numel() for p in model.parameters())
print("Model loaded strictly: {:.2f}M parameters".format(_n_params / 1e6))
print("  positional_len:", checkpoint_positional_len)
if _ckpt_epoch is not None:
    print("  epoch:", _ckpt_epoch)
if _metric_name is not None:
    print("  {}={}".format(_metric_name, _metric_val))

# Free CPU copy
del state_dict
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

Falling back to weights_only=False...
Tokenizer SHA-256 OK.
Model loaded strictly: 60.52M parameters
  positional_len: 192
  epoch: 17
  best_dev_chrf=40.72239756199789


In [6]:
# ── Effective decode length ───────────────────────────────────────────────
_positional_limit = int(model.pos_enc.pe.size(1))
EFFECTIVE_MAX_DECODE_TOKENS = MAX_DECODE_TOKENS if MAX_DECODE_TOKENS else max(1, MAX_LENGTH - 1)
EFFECTIVE_MAX_DECODE_TOKENS = min(EFFECTIVE_MAX_DECODE_TOKENS, _positional_limit - 1)
print("Effective max decode tokens:", EFFECTIVE_MAX_DECODE_TOKENS)

# ── Greedy decoder (debugging baseline) ──────────────────────────────────
@torch.inference_mode()
def greedy_translate(model_, src_text, max_len=None):
    """Greedy decoder for debugging. Returns empty string on error."""
    try:
        src_text = str(src_text).strip()
        if not src_text:
            return ""
        _max = max_len or EFFECTIVE_MAX_DECODE_TOKENS
        _pieces = sp.encode(src_text, add_bos=False, add_eos=False)[:max(0, MAX_LENGTH - 2)]
        _src_ids = [BOS_ID] + _pieces + [EOS_ID]
        src = torch.tensor([_src_ids], dtype=torch.long, device=device)
        _scale = model_.d_model ** 0.5
        _src_emb = model_.pos_enc(model_.embedding(src) * _scale)
        _memory  = model_.transformer.encoder(
            _src_emb, src_key_padding_mask=model_.pad_mask(src))
        _tgt_ids = [BOS_ID]
        for _ in range(_max):
            _tgt = torch.tensor([_tgt_ids], dtype=torch.long, device=device)
            _tgt_emb = model_.pos_enc(model_.embedding(_tgt) * _scale)
            _dec_out = model_.transformer.decoder(
                _tgt_emb, _memory,
                tgt_mask=model_.causal_mask(_tgt.size(1), device),
                memory_key_padding_mask=model_.pad_mask(src),
            )
            _logits = model_.output_proj(_dec_out[:, -1, :])
            _logits[:, PAD_ID] = float('-inf')
            _logits[:, BOS_ID] = float('-inf')
            _next_id = int(_logits.argmax(dim=-1).item())
            if _next_id == EOS_ID:
                break
            _tgt_ids.append(_next_id)
        _out_ids = [t for t in _tgt_ids if t not in (BOS_ID, EOS_ID, PAD_ID)]
        return sp.decode(_out_ids).strip()
    except Exception as e:
        return ""

# ── Beam search (correct sequential, no early termination bug) ────────────
@torch.inference_mode()
def beam_search(model_, src_text, beam_size=BEAM_SIZE, max_new_tokens=None,
                length_penalty=LENGTH_PENALTY, min_tokens=MIN_DECODE_TOKENS):
    """
    Correct sequential beam search.
    
    - Source: BOS + pieces[:MAX_LENGTH-2] + EOS
    - Encoder memory computed ONCE, expanded for active beams
    - Each beam: (cumulative_log_prob, token_list, is_finished)
    - Finished beams carried forward unchanged; only unfinished beams expanded
    - PAD and BOS never generated; EOS blocked for steps < min_tokens
    - Continues until ALL beams finish OR max_new_tokens reached
      (does NOT terminate just because beam_size finished beams exist)
    - Length penalty: score / ((5 + len_excl_bos) / 6)^alpha
    - Fallback to best unfinished beam if no finished beam exists
    - Strips BOS/EOS/PAD before sp.decode()
    - Returns empty string on error (never raises)
    """
    try:
        src_text = str(src_text).strip()
        if not src_text:
            return ""
        _mnt = max_new_tokens if max_new_tokens is not None else EFFECTIVE_MAX_DECODE_TOKENS
        _pieces = sp.encode(src_text, add_bos=False, add_eos=False)[:max(0, MAX_LENGTH - 2)]
        _src_ids = [BOS_ID] + _pieces + [EOS_ID]
        src = torch.tensor([_src_ids], dtype=torch.long, device=device)
        _src_pad_mask = model_.pad_mask(src)
        _scale = model_.d_model ** 0.5
        _src_emb = model_.pos_enc(model_.embedding(src) * _scale)
        _memory  = model_.transformer.encoder(
            _src_emb, src_key_padding_mask=_src_pad_mask)
        # beams: (log_prob, token_ids, is_finished)
        beams: List[Tuple[float, List[int], bool]] = [(0.0, [BOS_ID], False)]

        for step in range(_mnt):
            # Collect only unfinished beams for expansion
            _active = [(i, sc, seq) for i, (sc, seq, fin) in enumerate(beams) if not fin]
            if not _active:
                break  # ALL beams finished

            _tgt = torch.tensor([seq for _, _, seq in _active], dtype=torch.long, device=device)
            _n   = _tgt.size(0)
            _mem_exp  = _memory.expand(_n, -1, -1)
            _mask_exp = _src_pad_mask.expand(_n, -1)

            _tgt_emb = model_.pos_enc(model_.embedding(_tgt) * _scale)
            _dec_out = model_.transformer.decoder(
                _tgt_emb, _mem_exp,
                tgt_mask=model_.causal_mask(_tgt.size(1), device),
                memory_key_padding_mask=_mask_exp,
            )
            _lp = F.log_softmax(model_.output_proj(_dec_out[:, -1, :]), dim=-1)
            _lp[:, PAD_ID] = float('-inf')
            _lp[:, BOS_ID] = float('-inf')
            if step < min_tokens:
                _lp[:, EOS_ID] = float('-inf')

            # Start next candidates with already-finished beams (carry forward unchanged)
            _cands: List[Tuple[float, List[int], bool]] = [
                b for b in beams if b[2]
            ]

            _topk = min(beam_size, _lp.size(-1))
            _vals, _ids = torch.topk(_lp, k=_topk, dim=-1)

            for _row, (_, _old_sc, _old_seq) in enumerate(_active):
                for _tv, _ti in zip(_vals[_row].tolist(), _ids[_row].tolist()):
                    if not math.isfinite(_tv):
                        continue
                    _new_seq = _old_seq + [int(_ti)]
                    _new_sc  = float(_old_sc + _tv)
                    _fin     = int(_ti) == EOS_ID
                    _cands.append((_new_sc, _new_seq, _fin))

            if not _cands:
                break

            def _norm_score(b):
                _log, _seq, _ = b
                _gen_len = max(1, len(_seq) - 1)  # exclude BOS
                if length_penalty == 0:
                    return _log
                return _log / ((5.0 + _gen_len) / 6.0) ** length_penalty

            _cands.sort(key=_norm_score, reverse=True)
            beams = _cands[:beam_size]
            # NOTE: do NOT break here just because some beams are finished.
            # Continue until ALL beams are finished or max_new_tokens reached.

        _finished = [b for b in beams if b[2]]
        _pool = _finished if _finished else beams
        _best = max(_pool, key=_norm_score)
        _out_ids = [t for t in _best[1] if t not in (BOS_ID, EOS_ID, PAD_ID)]
        return sp.decode(_out_ids).strip()
    except Exception:
        return ""

# Smoke test
_smoke = "Scientists announced a new discovery about climate change."
_hyp_g = greedy_translate(model, _smoke)
_hyp_b = beam_search(model, _smoke)
print("Smoke test (greedy):", _hyp_g or "[EMPTY]")
print("Smoke test (beam)  :", _hyp_b or "[EMPTY]")
if not _hyp_b:
    print("WARNING: Beam smoke test returned empty. Check checkpoint.")

Effective max decode tokens: 127


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Smoke test (greedy): kompyuta iliyoangazwa ugunduzi mpya kuhusu mabadiliko ya hali ya hewa.
Smoke test (beam)  : Utafiti uliofanywa na ugunduzi mpya kuhusu mabadiliko ya hali ya hewa.


In [7]:
# ── Load a small slice of FLORES dev for health check ────────────────────
# (uses same data-loading logic as Cell 9 but is self-contained for early abort)
HEALTH_N = 40

def _load_flores_for_health():
    """Try to load the first HEALTH_N FLORES dev examples by any available method."""
    # 1. raw_flores_dev.json
    for _p in _all_matches("raw_flores_dev.json"):
        try:
            with open(_p, "r", encoding="utf-8") as _fh:
                _d = json.load(_fh)
            return list(_d["src"])[:HEALTH_N], list(_d["ref"])[:HEALTH_N]
        except Exception:
            continue
    # 2. Text files
    _src_files = _all_matches("eng_Latn.dev")
    _ref_files = _all_matches("swh_Latn.dev")
    for _sf in _src_files:
        for _rf in _ref_files:
            if _sf.parent == _rf.parent:
                try:
                    _s = [l.strip() for l in _sf.read_text("utf-8").splitlines() if l.strip()]
                    _r = [l.strip() for l in _rf.read_text("utf-8").splitlines() if l.strip()]
                    return _s[:HEALTH_N], _r[:HEALTH_N]
                except Exception:
                    continue
    # 3. cache_flores_dev.pt
    for _p in _all_matches("cache_flores_dev.pt"):
        try:
            _recs = safe_torch_load(_p, map_location="cpu")
            _specials = {PAD_ID, BOS_ID, EOS_ID}
            _src_h, _ref_h = [], []
            for _rec in _recs[:HEALTH_N]:
                _si = [int(t) for t in _rec["src"].tolist() if int(t) not in _specials]
                _ri = [int(t) for t in _rec["tgt"].tolist() if int(t) not in _specials]
                _src_h.append(sp.decode(_si).strip())
                _ref_h.append(sp.decode(_ri).strip())
            return _src_h, _ref_h
        except Exception:
            continue
    raise FileNotFoundError("Could not find any FLORES dev data for health check.")

_health_src, _health_ref = _load_flores_for_health()
print("Health check using {} examples.".format(len(_health_src)))

_h_greedy_hyps = []
_h_beam_hyps   = []
for _i, (_s, _r) in enumerate(zip(_health_src, _health_ref)):
    _g = greedy_translate(model, _s)
    _b = beam_search(model, _s)
    _h_greedy_hyps.append(_g)
    _h_beam_hyps.append(_b)
    if _i < 5:
        print("[{:02d}] SRC   : {}".format(_i, _s[:100]))
        print("      REF   : {}".format(_r[:100]))
        print("      GREEDY: {}".format(_g[:100] or "[EMPTY]"))
        print("      BEAM  : {}".format(_b[:100] or "[EMPTY]"))
        print()

# ── Compute health metrics ────────────────────────────────────────────────
_n = len(_h_beam_hyps)
_empty_pct       = sum(1 for h in _h_beam_hyps if not h.strip()) / _n
_imm_eos_pct     = sum(1 for h in _h_beam_hyps if not h.strip()) / _n
_avg_token_len   = sum(len(h.split()) for h in _h_beam_hyps) / _n
_unique_ratio    = len(set(h.strip() for h in _h_beam_hyps)) / _n
_ctr: Dict[str, int] = {}
for h in _h_beam_hyps:
    _ctr[h.strip()] = _ctr.get(h.strip(), 0) + 1
_top3 = sorted(_ctr.items(), key=lambda x: -x[1])[:3]
_greedy_beam_agree = sum(
    1 for g, b in zip(_h_greedy_hyps, _h_beam_hyps) if g.strip() == b.strip()) / _n

print("Health metrics ({} examples):".format(_n))
print("  empty_pct       : {:.3f}".format(_empty_pct))
print("  imm_eos_pct     : {:.3f}".format(_imm_eos_pct))
print("  avg_token_len   : {:.2f}".format(_avg_token_len))
print("  unique_ratio    : {:.3f}".format(_unique_ratio))
print("  greedy_beam_agree:{:.3f}".format(_greedy_beam_agree))
print("  top_3_common    :")
for _txt, _cnt in _top3:
    print("    ({}) '{}'".format(_cnt, _txt[:80] if _txt else "[EMPTY]"))

# ── Health gate ───────────────────────────────────────────────────────────
_health_fail_reasons = []
if _empty_pct > 0.20:
    _health_fail_reasons.append(">{:.0%} empty hypotheses".format(_empty_pct))
if _avg_token_len < 2.0:
    _health_fail_reasons.append("avg token length {:.1f} < 2".format(_avg_token_len))
if _unique_ratio < 0.05 and HEALTH_N > 10:
    _health_fail_reasons.append(
        "unique ratio {:.3f} < 0.05 (most hypotheses identical)".format(_unique_ratio))

if _health_fail_reasons:
    _msg = "HEALTH GATE FAILED:\n" + "\n".join("  - " + r for r in _health_fail_reasons)
    _msg += "\nThis checkpoint appears collapsed or incompatible."
    _msg += "\nSet ALLOW_FAILED_HEALTH_CHECK=True to bypass (debugging only)."
    if not ALLOW_FAILED_HEALTH_CHECK:
        raise RuntimeError(_msg)
    else:
        print("WARNING: Health gate bypassed. Results may be invalid.")
else:
    print("Health gate PASSED.")

# Save health results
_health_json = {
    "run_id": RUN_ID,
    "n": _n,
    "empty_pct": _empty_pct,
    "avg_token_len": _avg_token_len,
    "unique_ratio": _unique_ratio,
    "greedy_beam_agree": _greedy_beam_agree,
    "top_3_common": _top3,
    "health_gate_passed": len(_health_fail_reasons) == 0,
    "fail_reasons": _health_fail_reasons,
}
_health_out = RESULTS_DIR / "health_check_{}.json".format(RUN_ID)
_health_out.write_text(json.dumps(_health_json, indent=2, ensure_ascii=False), encoding="utf-8")
print("Health check saved:", _health_out)

Health check using 40 examples.
[00] SRC   : On Monday, scientists from the Stanford University School of Medicine announced the invention of a n
      REF   : Mnamo Jumatatu, wanasayansi kutoka Shule ya Tiba ya Chuo Kikuu cha Stanford walitangaza uvumbuzi wa 
      GREEDY: Siku ya Jumatatu, wanasayansi kutoka Chuo Kikuu cha Chuo Kikuu cha Kuandaa utekelezaji wa utekelezaj
      BEAM  : Siku ya Jumatatu, wanasayansi kutoka Chuo Kikuu cha Chuo Kikuu cha Kuandaa utekelezaji wa uhifadhi m

[01] SRC   : Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to pati
      REF   : Watafiti wakuu wanasema hili linaweza kuleta ugunduzi wa mapema wa saratani, kifua kikuu, ukimwi na 
      GREEDY: Leadoma watafiti wanaweza kuleta mhariri wa saratani za mapema, utunzaji wa saratani, na virusi vya 
      BEAM  : Leadoma watafiti wanaweza kuleta mhariri wa saratani za mapema, utunzaji wa kansa za UKIMWI, na kwa 

[02] SRC   : The JAS 39C Gripen crashed onto a r

In [8]:
# ── FLORES loading helpers ────────────────────────────────────────────────
def _validate_parallel(src, ref, split_name):
    _s = [str(x).strip() for x in src]
    _r = [str(x).strip() for x in ref]
    if not _s:
        raise ValueError("{} source list is empty".format(split_name))
    if len(_s) != len(_r):
        raise ValueError("{} length mismatch: {} src vs {} ref".format(
            split_name, len(_s), len(_r)))
    _bad_s = [i for i, x in enumerate(_s) if not x]
    _bad_r = [i for i, x in enumerate(_r) if not x]
    if _bad_s:
        raise ValueError("{} has empty src at indices {}".format(split_name, _bad_s[:5]))
    if _bad_r:
        raise ValueError("{} has empty ref at indices {}".format(split_name, _bad_r[:5]))
    return _s, _r

def _load_flores_split(split_name):
    """Load FLORES split in priority order. Returns (src, ref, origin_str)."""
    # 1. raw_flores_{split}.json
    for _p in sorted(_all_matches("raw_flores_{}.json".format(split_name)),
                     key=lambda p: (-_common_prefix_depth(p.parent, RUN_DIR), len(p.parts), str(p))):
        try:
            with open(_p, "r", encoding="utf-8") as _fh:
                _d = json.load(_fh)
            _s = _d.get("src", _d.get("source"))
            _r = _d.get("ref", _d.get("tgt", _d.get("reference")))
            if _s is None or _r is None:
                continue
            _s, _r = _validate_parallel(_s, _r, split_name)
            print("Loaded {} from raw JSON: {} ({} pairs)".format(split_name, _p, len(_s)))
            return _s, _r, "raw_json:{}".format(_p)
        except Exception as e:
            print("  Skipping {} ({})".format(_p, e))

    # 2. FLORES text files eng_Latn.{split} + swh_Latn.{split}
    _sf_all = _all_matches("eng_Latn.{}".format(split_name))
    _rf_all = _all_matches("swh_Latn.{}".format(split_name))
    _par_dir = {p.parent: p for p in _sf_all}
    _par_ref = {p.parent: p for p in _rf_all}
    _common  = set(_par_dir) & set(_par_ref)
    for _par in sorted(_common,
                       key=lambda p: (-_common_prefix_depth(p, RUN_DIR), len(p.parts), str(p))):
        try:
            _s = [l.strip() for l in _par_dir[_par].read_text("utf-8").splitlines() if l.strip()]
            _r = [l.strip() for l in _par_ref[_par].read_text("utf-8").splitlines() if l.strip()]
            _s, _r = _validate_parallel(_s, _r, split_name)
            print("Loaded {} from text files: {} ({} pairs)".format(split_name, _par, len(_s)))
            return _s, _r, "flores_text:{}".format(_par)
        except Exception as e:
            print("  Skipping text pair in {} ({})".format(_par, e))

    # 3. LAST RESORT: cache_flores_{split}.pt
    for _p in sorted(_all_matches("cache_flores_{}.pt".format(split_name)),
                     key=lambda p: (-_common_prefix_depth(p.parent, RUN_DIR), len(p.parts), str(p))):
        try:
            print("WARNING: Loading {} from tokenized cache {}.".format(split_name, _p))
            print("  STRONG WARNING: Scores computed from decoded cache may differ from"
                  " official FLORES text scores. Do not use for paper-comparable results.")
            _recs = safe_torch_load(_p, map_location="cpu")
            _specials = {PAD_ID, BOS_ID, EOS_ID}
            _s, _r = [], []
            for _rec in _recs:
                _si = [int(t) for t in _rec["src"].tolist() if int(t) not in _specials]
                _ri = [int(t) for t in _rec["tgt"].tolist() if int(t) not in _specials]
                _s.append(sp.decode(_si).strip())
                _r.append(sp.decode(_ri).strip())
            _s, _r = _validate_parallel(_s, _r, split_name)
            print("  Loaded {} pairs from cache.".format(len(_s)))
            return _s, _r, "decoded_cache:{}".format(_p)
        except Exception as e:
            print("  Cache load failed ({})".format(e))

    raise FileNotFoundError(
        "Could not find FLORES {} data by any method.".format(split_name))

# ── Expected sizes ────────────────────────────────────────────────────────
_EXPECTED = {"dev": 997, "devtest": 1012}

dev_src = dev_ref = devtest_src = devtest_ref = []

if EVALUATE_DEV:
    dev_src, dev_ref, _dev_origin = _load_flores_split("dev")
    if abs(len(dev_src) - _EXPECTED["dev"]) > 20:
        print("WARNING: dev has {} examples; expected ~{}.".format(
            len(dev_src), _EXPECTED["dev"]))
    print("FLORES dev : {} pairs (origin: {})".format(len(dev_src), _dev_origin))
    print("  First 3 examples:")
    for _i in range(min(3, len(dev_src))):
        print("  [{:d}] SRC: {}".format(_i, dev_src[_i][:100]))
        print("       REF: {}".format(dev_ref[_i][:100]))

if EVALUATE_DEVTEST:
    devtest_src, devtest_ref, _devtest_origin = _load_flores_split("devtest")
    if abs(len(devtest_src) - _EXPECTED["devtest"]) > 20:
        print("WARNING: devtest has {} examples; expected ~{}.".format(
            len(devtest_src), _EXPECTED["devtest"]))
    print("FLORES devtest: {} pairs (origin: {})".format(len(devtest_src), _devtest_origin))
    print("  First 3 examples:")
    for _i in range(min(3, len(devtest_src))):
        print("  [{:d}] SRC: {}".format(_i, devtest_src[_i][:100]))
        print("       REF: {}".format(devtest_ref[_i][:100]))

Loaded dev from text files: /kaggle/input/datasets/nirmitmistry/evalaution/beam-m1-eval/flores (997 pairs)
FLORES dev : 997 pairs (origin: flores_text:/kaggle/input/datasets/nirmitmistry/evalaution/beam-m1-eval/flores)
  First 3 examples:
  [0] SRC: On Monday, scientists from the Stanford University School of Medicine announced the invention of a n
       REF: Mnamo Jumatatu, wanasayansi kutoka Shule ya Tiba ya Chuo Kikuu cha Stanford walitangaza uvumbuzi wa 
  [1] SRC: Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to pati
       REF: Watafiti wakuu wanasema hili linaweza kuleta ugunduzi wa mapema wa saratani, kifua kikuu, ukimwi na 
  [2] SRC: The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closi
       REF: JAS 39C Gripen ilianguka kwenye barabara kuu karibu masaa ya 9:30 asubuhi (0230 UTC) na kulipuka, kw
Loaded devtest from text files: /kaggle/input/datasets/nirmitmistry/evalaution/beam-m1

In [9]:
# ── Decode tag and digest ─────────────────────────────────────────────────
_safe_lp = str(LENGTH_PENALTY).replace(".", "p")
DECODE_TAG = "beam{}_lp{}_max{}".format(BEAM_SIZE, _safe_lp, EFFECTIVE_MAX_DECODE_TOKENS)
print("DECODE_TAG:", DECODE_TAG)

def _source_digest(src_list):
    return hashlib.sha256("\n".join(src_list).encode("utf-8")).hexdigest()[:16]

def _load_partial_predictions(partial_path, src_list, source_digest, decode_tag):
    """
    Load previously saved partial predictions if they exist and are compatible.
    Verifies each row's index, source_digest, and src text.
    Truncates to last valid row if final row was incomplete.
    Returns list of hypotheses for completed rows, or [] if no valid file.
    """
    if not partial_path.exists() or not RESUME_PARTIAL:
        return []
    _valid: List[Dict] = []
    try:
        with open(partial_path, "r", encoding="utf-8") as _fh:
            for _lineno, _line in enumerate(_fh, start=1):
                _line = _line.strip()
                if not _line:
                    continue
                try:
                    _row = json.loads(_line)
                except json.JSONDecodeError:
                    print("Ignoring incomplete final line in {}".format(partial_path.name))
                    break
                _exp_idx = len(_valid)
                if _row.get("index") != _exp_idx:
                    raise ValueError(
                        "Partial file index mismatch at line {}: expected {}, got {}".format(
                            _lineno, _exp_idx, _row.get("index")))
                if _row.get("source_digest") != source_digest:
                    raise ValueError(
                        "Partial file {} belongs to a different evaluation source".format(
                            partial_path))
                if (_exp_idx >= len(src_list) or
                        _row.get("src") != src_list[_exp_idx]):
                    raise ValueError(
                        "Partial file source mismatch at index {}".format(_exp_idx))
                _valid.append(_row)
    except Exception as exc:
        print("WARNING: Could not resume partial file: {}".format(exc))
        return []

    # Rewrite only validated rows
    _tmp = partial_path.with_suffix(".tmp")
    with open(_tmp, "w", encoding="utf-8") as _fh:
        for _row in _valid:
            _fh.write(json.dumps(_row, ensure_ascii=False) + "\n")
    _tmp.replace(partial_path)

    _hyps = [str(_row["hyp"]) for _row in _valid]
    if _hyps:
        print("Resuming {}: {} translations already done.".format(
            partial_path.name, len(_hyps)))
    return _hyps

def run_evaluation(split_name, src_list, ref_list):
    """
    Translate src_list using beam_search with RESUME_PARTIAL support.
    Saves partial JSONL every 10 translations.
    Computes BLEU and chrF++ at end.
    Returns dict with src, ref, hyp, bleu, chrf_pp, bleu_signature, chrf_signature, partial_path.
    """
    _src = list(src_list)
    _ref = list(ref_list)
    if EVAL_LIMIT is not None:
        _src = _src[:EVAL_LIMIT]
        _ref = _ref[:EVAL_LIMIT]
        print("WARNING: EVAL_LIMIT={} applied. Not a full-set result.".format(EVAL_LIMIT))

    _src, _ref = _validate_parallel(_src, _ref, split_name)
    _digest = _source_digest(_src)

    _partial_path = PREDS_DIR / ".{}_{}_{}_{}_partial.jsonl".format(
        RUN_ID, split_name, DECODE_TAG, _digest[:8])

    _hyps = _load_partial_predictions(_partial_path, _src, _digest, DECODE_TAG)
    _start = len(_hyps)

    _mode = "a" if _hyps else "w"
    with open(_partial_path, _mode, encoding="utf-8", buffering=1) as _pfh:
        _pbar = tqdm(range(_start, len(_src)), total=len(_src),
                     initial=_start, desc="FLORES {}".format(split_name))
        for _i in _pbar:
            _h = beam_search(model, _src[_i])
            _hyps.append(_h)
            _row = {"index": _i, "source_digest": _digest,
                    "src": _src[_i], "hyp": _h}
            _pfh.write(json.dumps(_row, ensure_ascii=False) + "\n")
            if (_i + 1) % 10 == 0:
                _pfh.flush()

    _bleu_metric = sacrebleu.metrics.BLEU(tokenize=BLEU_TOKENIZER)
    _chrf_metric = sacrebleu.metrics.CHRF(word_order=2)
    _bleu_res = _bleu_metric.corpus_score(_hyps, [_ref])
    _chrf_res = _chrf_metric.corpus_score(_hyps, [_ref])

    return {
        "split":          split_name,
        "src":            _src,
        "ref":            _ref,
        "hyp":            _hyps,
        "bleu":           float(_bleu_res.score),
        "chrf_pp":        float(_chrf_res.score),
        "bleu_signature": str(_bleu_metric.get_signature()),
        "chrf_signature": str(_chrf_metric.get_signature()),
        "partial_path":   _partial_path,
    }

DECODE_TAG: beam5_lp0p6_max127


In [10]:
evaluation_results: Dict[str, Dict] = {}

if EVALUATE_DEV and len(dev_src) > 0:
    print("Evaluating on FLORES dev ({} examples)...".format(len(dev_src)))
    evaluation_results["dev"] = run_evaluation("dev", dev_src, dev_ref)
    _r = evaluation_results["dev"]
    print("FLORES dev   BLEU={:.2f}  chrF++={:.2f}".format(_r["bleu"], _r["chrf_pp"]))
    print("  Signature:", _r["bleu_signature"])

if EVALUATE_DEVTEST and len(devtest_src) > 0:
    print("Evaluating on FLORES devtest ({} examples)...".format(len(devtest_src)))
    evaluation_results["devtest"] = run_evaluation("devtest", devtest_src, devtest_ref)
    _r = evaluation_results["devtest"]
    print("FLORES devtest  BLEU={:.2f}  chrF++={:.2f}".format(_r["bleu"], _r["chrf_pp"]))
    print("  Signature:", _r["bleu_signature"])

if not evaluation_results:
    print("WARNING: No evaluation was run. Check EVALUATE_DEV/EVALUATE_DEVTEST settings.")

Evaluating on FLORES dev (997 examples)...


FLORES dev:   0%|          | 0/997 [00:00<?, ?it/s]

FLORES dev   BLEU=14.14  chrF++=40.31
  Signature: nrefs:1|case:mixed|eff:no|tok:flores200|smooth:exp|version:2.6.0
Evaluating on FLORES devtest (1012 examples)...


FLORES devtest:   0%|          | 0/1012 [00:00<?, ?it/s]

FLORES devtest  BLEU=13.45  chrF++=39.81
  Signature: nrefs:1|case:mixed|eff:no|tok:flores200|smooth:exp|version:2.6.0


In [11]:
import tempfile

def atomic_write(path, text):
    """Write text to path atomically using a temp file + os.replace."""
    _fd, _tmp = tempfile.mkstemp(dir=str(Path(path).parent), suffix='.tmp')
    try:
        os.close(_fd)
        with open(_tmp, 'w', encoding='utf-8') as _f:
            _f.write(text)
        os.replace(_tmp, str(path))
    except Exception:
        try:
            os.unlink(_tmp)
        except Exception:
            pass
        raise

# ── Save predictions for each split ──────────────────────────────────────
for _split, _result in evaluation_results.items():
    _hyp_file = PREDS_DIR / "{}_{}_{}hyp.txt".format(RUN_ID, _split, DECODE_TAG + "_")
    _csv_file = PREDS_DIR / "{}_{}_{}preds.csv".format(RUN_ID, _split, DECODE_TAG + "_")

    # .txt: one hypothesis per line
    atomic_write(_hyp_file, "\n".join(_result["hyp"]))
    print("Saved hypotheses:", _hyp_file)

    # .csv: index, source, reference, hypothesis
    _df = pd.DataFrame({
        "index":     list(range(len(_result["src"]))),
        "source":    _result["src"],
        "reference": _result["ref"],
        "hypothesis":_result["hyp"],
    })
    atomic_write(_csv_file, _df.to_csv(index=False))
    print("Saved CSV:", _csv_file)

    # Delete partial JSONL after successful final save
    _pp = _result.get("partial_path")
    if _pp and Path(_pp).exists():
        Path(_pp).unlink()
        print("Removed partial file:", Path(_pp).name)

# ── Append to student_all_scores.csv ─────────────────────────────────────
_scores_csv = RESULTS_DIR / "student_all_scores.csv"
_score_rows = []
for _split, _result in evaluation_results.items():
    _score_rows.append({
        "model":          RUN_ID,
        "dataset":        DATASET,
        "model_size":     MODEL_SIZE,
        "eval_set":       _split,
        "beam_size":      BEAM_SIZE,
        "length_penalty": LENGTH_PENALTY,
        "bleu":           _result["bleu"],
        "chrf_pp":        _result["chrf_pp"],
        "n_examples":     len(_result["src"]),
        "bleu_signature": _result["bleu_signature"],
        "chrf_signature": _result["chrf_signature"],
    })

_new_df = pd.DataFrame(_score_rows)
if _scores_csv.exists():
    _old_df = pd.read_csv(_scores_csv)
    _combined = pd.concat([_old_df, _new_df], ignore_index=True)
    _combined = _combined.drop_duplicates(
        subset=["model", "eval_set", "beam_size", "length_penalty"], keep="last")
else:
    _combined = _new_df
atomic_write(_scores_csv, _combined.to_csv(index=False))
print("Updated:", _scores_csv)

# ── Save metrics JSON ─────────────────────────────────────────────────────
_metrics = {
    "run_id":        RUN_ID,
    "checkpoint":    str(CKPT_BEST),
    "dataset":       DATASET,
    "model_size":    MODEL_SIZE,
    "decode_tag":    DECODE_TAG,
    "beam_size":     BEAM_SIZE,
    "length_penalty":LENGTH_PENALTY,
    "vocab_size":    VOCAB_SIZE,
    "max_length":    MAX_LENGTH,
    "scores": {
        _split: {
            "bleu":           _r["bleu"],
            "chrf_pp":        _r["chrf_pp"],
            "n_examples":     len(_r["src"]),
            "bleu_signature": _r["bleu_signature"],
            "chrf_signature": _r["chrf_signature"],
        }
        for _split, _r in evaluation_results.items()
    },
}
_metrics_json = RESULTS_DIR / "metrics_{}.json".format(RUN_ID)
atomic_write(_metrics_json, json.dumps(_metrics, indent=2, ensure_ascii=False))
print("Saved metrics:", _metrics_json)

# ── Save run manifest ─────────────────────────────────────────────────────
import datetime
_manifest = {
    "run_id":         RUN_ID,
    "timestamp":      datetime.datetime.utcnow().isoformat() + "Z",
    "checkpoint":     str(CKPT_BEST),
    "spm_model":      str(SPM_MODEL_PATH),
    "vocab_info":     str(VOCAB_INFO_PATH),
    "decode_tag":     DECODE_TAG,
    "eval_limit":     EVAL_LIMIT,
    "health_passed":  len(_health_fail_reasons) == 0,
    "scores":         {_s: {"bleu": _r["bleu"], "chrf_pp": _r["chrf_pp"]}
                       for _s, _r in evaluation_results.items()},
}
_manifest_json = RESULTS_DIR / "manifest_{}.json".format(RUN_ID)
atomic_write(_manifest_json, json.dumps(_manifest, indent=2, ensure_ascii=False))
print("Saved manifest:", _manifest_json)

# ── Print sample translations ─────────────────────────────────────────────
_primary = "dev" if "dev" in evaluation_results else (list(evaluation_results.keys())[0] if evaluation_results else None)
if _primary:
    print("\n--- 5 sample translations ({}) ---".format(_primary))
    _r = evaluation_results[_primary]
    for _i in range(min(5, len(_r["src"]))):
        print("[{:d}] SRC: {}".format(_i, _r["src"][_i][:120]))
        print("     REF: {}".format(_r["ref"][_i][:120]))
        print("     HYP: {}".format(_r["hyp"][_i][:120] or "[EMPTY]"))
        print()

Saved hypotheses: /kaggle/working/results/predictions/student_beam_M10_optA_fixed_v2_best_chrf_dev_beam5_lp0p6_max127_hyp.txt
Saved CSV: /kaggle/working/results/predictions/student_beam_M10_optA_fixed_v2_best_chrf_dev_beam5_lp0p6_max127_preds.csv
Removed partial file: .student_beam_M10_optA_fixed_v2_best_chrf_dev_beam5_lp0p6_max127_2537b856_partial.jsonl
Saved hypotheses: /kaggle/working/results/predictions/student_beam_M10_optA_fixed_v2_best_chrf_devtest_beam5_lp0p6_max127_hyp.txt
Saved CSV: /kaggle/working/results/predictions/student_beam_M10_optA_fixed_v2_best_chrf_devtest_beam5_lp0p6_max127_preds.csv
Removed partial file: .student_beam_M10_optA_fixed_v2_best_chrf_devtest_beam5_lp0p6_max127_8211a301_partial.jsonl
Updated: /kaggle/working/results/student_all_scores.csv
Saved metrics: /kaggle/working/results/metrics_student_beam_M10_optA_fixed_v2_best_chrf.json
Saved manifest: /kaggle/working/results/manifest_student_beam_M10_optA_fixed_v2_best_chrf.json

--- 5 sample translations (de

/tmp/ipykernel_58/2277305079.py:102: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":      datetime.datetime.utcnow().isoformat() + "Z",


In [12]:
# ── Optional teacher comparison ───────────────────────────────────────────
_teacher_csv_cands = _all_matches("teacher_flores_scores.csv")
if _teacher_csv_cands:
    _teacher_csv = sorted(
        _teacher_csv_cands,
        key=lambda p: (len(p.parts), str(p))
    )[0]
    try:
        _tdf = pd.read_csv(_teacher_csv)
        print("Teacher scores loaded from:", _teacher_csv)
        print(_tdf.to_string(index=False))
        print()

        # Find beam M=1 row
        _beam_m1_rows = _tdf[
            (_tdf.get("beam_size", pd.Series(dtype=object)) == 1) |
            (_tdf.get("method", pd.Series(dtype=object)).astype(str).str.contains("beam_M1", na=False))
        ] if not _tdf.empty else pd.DataFrame()

        if not _beam_m1_rows.empty:
            _teacher_row = _beam_m1_rows.iloc[0]
            print("Teacher (beam M=1) comparison:")
            for _split, _res in evaluation_results.items():
                _bleu_col  = "bleu" if "bleu"   in _tdf.columns else None
                _chrf_col  = "chrf" if "chrf"   in _tdf.columns else (
                             "chrf_pp" if "chrf_pp" in _tdf.columns else None)
                if _bleu_col:
                    _t_bleu = float(_teacher_row[_bleu_col])
                    _s_bleu = _res["bleu"]
                    print("  {} BLEU  : student={:.2f}  teacher={:.2f}  gap={:+.2f}".format(
                        _split, _s_bleu, _t_bleu, _s_bleu - _t_bleu))
                if _chrf_col:
                    _t_chrf = float(_teacher_row[_chrf_col])
                    _s_chrf = _res["chrf_pp"]
                    print("  {} chrF++: student={:.2f}  teacher={:.2f}  gap={:+.2f}".format(
                        _split, _s_chrf, _t_chrf, _s_chrf - _t_chrf))
        else:
            print("Could not identify beam_M1 row in teacher scores for comparison.")
    except Exception as _e:
        print("Could not load teacher scores: {}".format(_e))
else:
    print("No teacher_flores_scores.csv found; skipping teacher comparison.")

# ── Print student_all_scores.csv ──────────────────────────────────────────
print("\n--- All student scores ---")
if _scores_csv.exists():
    _all_scores = pd.read_csv(_scores_csv)
    print(_all_scores.to_string(index=False))
else:
    print("(No student_all_scores.csv found)")

print("\n" + "="*60)
print("Evaluation complete.")
print("  Run ID   :", RUN_ID)
print("  Checkpoint:", CKPT_BEST)
for _split, _res in evaluation_results.items():
    print("  {} BLEU={:.2f}  chrF++={:.2f}".format(
        _split, _res["bleu"], _res["chrf_pp"]))
print("  Results dir:", RESULTS_DIR)
print("="*60)

No teacher_flores_scores.csv found; skipping teacher comparison.

--- All student scores ---
                                   model  dataset model_size eval_set  beam_size  length_penalty      bleu   chrf_pp  n_examples                                                   bleu_signature                                              chrf_signature
student_beam_M10_optA_fixed_v2_best_chrf beam_M10          A      dev          5             0.6 14.140974 40.306836         997 nrefs:1|case:mixed|eff:no|tok:flores200|smooth:exp|version:2.6.0 nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|version:2.6.0
student_beam_M10_optA_fixed_v2_best_chrf beam_M10          A  devtest          5             0.6 13.453176 39.810569        1012 nrefs:1|case:mixed|eff:no|tok:flores200|smooth:exp|version:2.6.0 nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|version:2.6.0

Evaluation complete.
  Run ID   : student_beam_M10_optA_fixed_v2_best_chrf
  Checkpoint: /kaggle/input/datasets/nirmitmistry/beamm10/beam_M10/st